In [ ]:
using DelimitedFiles
include("open_optimization_problem.jl")   # pulls in the full include chain

n      = 3
J      = fill(1/4, n - 1)
gammas = fill(J[1]/4, n)
tlist  = range(0, 25; length = 100)
excited = ["0"]
ks     = [3, 8, 12]
dissipation = false

cutoff, maxdim = 0.0, 16     # was 1e-10, 200 — maxdim=16 is already exact for n=3
order     = 2     # order of the product formulas being combined
k_ref     = 100   # fine reference standing in for e^{tL} in F_ex
order_ref = 4                # was 2

lsites = liouville_siteinds(n)
rho0   = vectorized_initial_state_mps(lsites, excited)

coeffs = zeros(Float64, length(tlist), length(ks))

for (i, t) in enumerate(tlist)
    if t <= 0
        coeffs[i, :] .= NaN
        continue
    end
    M, _ = open_gram_matrix(n, J, gammas, t, ks, lsites, rho0;
                            cutoff = cutoff, maxdim = maxdim,
                            order = order, dissipation = dissipation)
    L, _ = open_L_vector(n, J, gammas, t, ks, k_ref, lsites, rho0;
                         cutoff = cutoff, maxdim = maxdim,
                         order = order, order_ref = order_ref,
                         dissipation = dissipation)
    c, _ = dynamic_mpf_coefficients(M, L)
    coeffs[i, :] .= c
    println("t = ", round(t, digits = 4), "  c = ", c,
            "  sum = ", sum(c), "  cond(M) = ", cond(M))
end

open("n_3_mpf_coefficients.txt", "w") do io
    println(io, "# t\tc_k3\tc_k8\tc_k12")
    writedlm(io, hcat(collect(tlist), coeffs))
end